# 05 – Evaluation Results
Score both systems and compare performance.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, '..')

from src.evaluation.scorer import load_config, score_all
from src.evaluation.metrics import coverage_curve, print_summary_table, plot_comparison

cfg = load_config('../config.yaml')
GT_DIR = '../data/processed/critique_dicts'

In [ ]:
print('=== BASELINE ===')
baseline_scores = score_all('../results/baseline', GT_DIR, cfg)
with open('../results/baseline/scores.json', 'w') as f:
    json.dump(baseline_scores, f, indent=2)

print('\n=== AGENT ===')
agent_scores = score_all('../results/agents', GT_DIR, cfg)
with open('../results/agents/scores.json', 'w') as f:
    json.dump(agent_scores, f, indent=2)

In [ ]:
print_summary_table(baseline_scores, agent_scores)

In [ ]:
plot_comparison(baseline_scores, agent_scores, output_path='../results/comparison.png')

In [ ]:
# Precision-Recall curves across thresholds
import matplotlib.pyplot as plt

b_curve = coverage_curve('../results/baseline', GT_DIR, cfg)
a_curve = coverage_curve('../results/agents', GT_DIR, cfg)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, metric, title in [
    (axes[0], 'recall',    'Recall @ threshold'),
    (axes[1], 'precision', 'Precision @ threshold'),
]:
    ax.plot(b_curve['threshold'], b_curve[metric], 'o-', label='Baseline')
    ax.plot(a_curve['threshold'], a_curve[metric], 's-', label='Agent')
    ax.set_xlabel('Similarity threshold')
    ax.set_ylabel(metric.capitalize())
    ax.set_title(title)
    ax.legend()
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('../results/coverage_curves.png', dpi=150)
plt.show()